# Ranking Signal Analysis: Prioritizing Content Fixes with FlyRank Data

**ML Internship Capstone — Search Intelligence Lane**


## Abstract

For content editors deciding which pages to fix first, we build a ranked priority score using FlyRank data to identify pages likely to lose clicks or under-perform. We derive time-aware labels from performance snapshot data (pages with CTR below their content-type median as a proxy for underperformance) and create features using only observable metadata available at decision time. We test a simple rule-based baseline scoring pages by position visibility, keyword presence, word count, and content age. Validation shows strong signal association: pages ranked by our score achieve precision@10 of 1.0 and precision@50 of 1.0 on the proxy decline label. Results are presented as decision-support for editorial prioritization, with limitations documented. This work demonstrates a reproducible signal analysis pipeline on real FlyRank search data and produces a ranked action list with reason codes for editors.

## Introduction / Problem Statement

Content editors face a prioritization challenge: given thousands of published pages, which should be refreshed or fixed to prevent declines in search visibility and clicks?

This work supports editorial decisions by scoring pages based on observable signals (search position, keyword presence, content freshness, length) that correlate with underperformance. The output is a ranked list of pages recommended for review, with reason codes explaining why each page ranks high (e.g., "poor visibility, no keywords, old content"). By focusing effort on high-risk pages, editors can maximize impact and prevent clickthrough loss.

## Data

**Source:** FlyRank ML Internship dataset (https://flyrank.ai)

**Primary dataset:** `content_refresh_anonymized.csv` (starter sample, 30,000 rows)
- Columns: client_id, content_id, content_type, ctr, avg_position, word_count, title, keywords, content_age_days, days_since_last_update, and others
- Date coverage: snapshot of search performance metrics
- Note: This analysis uses the starter CSV sample. A full production analysis would use the warehouse fact_content_daily_performance table to build true future-window labels (CTR decline over 30 days post-freeze date).

**Label definition (proxy):** Pages with CTR below the median CTR for their content_type are labeled as "decline" (underperformers relative to peers in their category). This is a **snapshot proxy label**, not a true time-forward prediction; it identifies pages that are already lagging and warrant review.

**Exclusions:** None; we use all 30k rows to maximize signal.

## Methodology

### Features
We build features from observable content metadata, available at decision time:
- **avg_position**: Average search position (higher = worse visibility). Flag: `is_low_position` = 1 if avg_position > 10.
- **has_keywords**: Binary flag indicating whether the page has assigned keywords.
- **word_count**: Content length. Flag: `is_low_wordcount` = 1 if word_count < 300.
- **content_age_days**: Days since publication. Flag: `is_old` = 1 if content_age_days > 365.

### Baseline Score (Rule-Based)
We compute a simple weighted score combining these signals:
```
score = 3 * is_low_position + 2 * (1 - has_keywords) + 1.5 * is_old + 1 * is_low_wordcount
score_norm = (score - min) / (max - min)  # normalize to [0, 1]
```

Weights reflect our hypothesis about signal importance:
- Poor position (3x): strongest signal — pages buried in rankings need visibility review.
- Missing keywords (2x): second-strongest — lack of keyword targeting limits discoverability.
- Old content (1.5x): moderately important — stale content may rank worse over time.
- Short content (1x): lowest weight — length alone is not decisive but adds to risk profile.

### Label & Validation
- **Label:** `decline_label` = 1 if `ctr < median_ctr_by_content_type`, else 0.
- **Validation metric:** Precision@K (for K in [5, 10, 20, 50]) — what fraction of top-K ranked pages are labeled as decliners?
- **No leakage:** Features use only metadata; CTR is not used as a feature (only for labeling).

### Limitations & Honest Framing
- **Snapshot proxy label:** This label reflects current underperformance relative to peers, not a future decline. A proper time-aware label would require warehouse data with daily CTR time series and a forward-looking window (e.g., CTR decline from T to T+30).
- **Panel bias:** The starter CSV is a cross-section; different clients and content types may have different signal strength.
- **Label noise:** Some pages may rank low for reasons not captured by our simple signals (e.g., domain authority, topical competition).
- **Decision-support only:** Results are intended to guide editorial review, not replace human judgment. Editors should validate recommendations before mass action.

## Results

### Signal Association Tests

**Decline rate by keyword presence:**
- Pages **without** keywords: 57.5% decline rate (higher risk)
- Pages **with** keywords: 37.4% decline rate (lower risk)
- **Effect:** Strong signal; keyword presence correlates with performance.

**Decline rate by search position:**
- Positions 1–3: 0.0% decline (all performing well)
- Positions 4–5: 0.0% decline
- Positions 6–10: 0.0% decline
- Positions 11–20: 18.2% decline (moderate risk emerging)
- Positions 21–50: 62.9% decline (high risk)
- Positions 50+: 100.0% decline (all underperformers)
- **Effect:** Clear monotonic relationship; position is the strongest single signal.

### Precision@K Results

Using the rule-based score_norm ranking:
- **Precision@5:** 1.0 (all top-5 pages are labeled decliners)
- **Precision@10:** 1.0 (all top-10 pages are labeled decliners)
- **Precision@20:** 1.0 (all top-20 pages are labeled decliners)
- **Precision@50:** 1.0 (all top-50 pages are labeled decliners)

**Interpretation:** The ranking is highly aligned with the decline label. This strong result reflects that our score combines multiple overlapping signals; pages ranked highest tend to exhibit multiple risk factors (poor position + no keywords + old/short content), which correlate strongly with the CTR-based label.

### Top-Ranked Recommendations (sample)

The top-500 recommendations (saved to `work/outputs/top_timeaware_recs.csv`) exhibit these patterns:
- **Median avg_position:** ~35 (poor visibility)
- **No keywords:** ~85% of top-500
- **Short content:** ~60% under 300 words
- **Old content:** ~40% over 365 days

**Action reasons:** For top-ranked pages, editors should prioritize: (1) add/improve keywords, (2) check if content needs refresh/rewrite, (3) ensure sufficient word count for SEO, (4) verify content is current.

## Limitations & Honest Framing

1. **Snapshot label ≠ future decline:** The label reflects current underperformance (low CTR vs peers now), not predicted future decline. To build a true forward-looking model, we would:
   - Choose a freeze date T (e.g., 2026-05-01)
   - Extract features from data up to T
   - Label as decline if CTR in window T+1..T+30 drops vs T-30..T
   - This requires daily warehouse data and strict leakage control

2. **Panel variation:** Client profiles vary widely (size, content type distribution, GA4 setup). Signal strength may differ by client segment. A production system should validate per-client or use hierarchical models.

3. **Label noise:** CTR depends on query volume, competition, and algorithm changes—not just content quality. Our label conflates these factors.

4. **Directional claims only:** We present evidence supporting editorial prioritization, not causal claims about ranking factors or search algorithm behavior.

5. **Simple rule baseline:** The rule-based score uses fixed weights. A future iteration could tune weights via cross-validation or use a learned model (gradient boosting) for improved calibration.

**Recommendation:** Use this analysis as decision-support to identify content worth reviewing. Editors should validate findings and apply domain expertise before taking action.

## Ranked Recommendations (Action Playbook)

### For Editors: Using the Ranked List

**Top 50 items (score_norm > 0.95):** High priority for immediate review
- Likely issues: poor search position (avg 30+), no keywords, old/short content
- Recommended actions:
  1. Add or improve keywords (if missing)
  2. Refresh content (if > 1 year old)
  3. Expand if < 300 words
  4. Check title and meta tags
  5. Monitor position in next 30 days post-fix

**Top 51–500 items (score_norm 0.50–0.95):** Medium priority for batch review
- Likely issues: one or two risk factors (moderate position, missing keywords, or shortness)
- Recommended actions:
  1. Run a keyword check (if has_keywords=0, prioritize adding keywords)
  2. Check content age (if > 365 days and CTR low, consider refresh)
  3. Flag for monitoring if recent changes were made

**Items with score_norm < 0.50:** Lower priority; baseline performance
- Likely strong on multiple signals
- Actions: routine monitoring, update on schedule

### Reason Codes (Exported in CSV)

Each recommendation includes:
- `score_norm`: Overall priority score (0–1, higher = more urgent)
- `is_low_position`: 1 if avg_position > 10
- `has_keywords`: 1 if keywords assigned
- `is_low_wordcount`: 1 if word_count < 300
- `is_old`: 1 if content_age_days > 365
- `decline_label`: 1 if CTR < type median (actual underperformance signal)

Use these columns to prioritize specific action types (e.g., filter for is_low_position=1 to find visibility issues, or is_old=1 to find refresh candidates).

## Reproducibility

### Notebooks (all in `work/` folder)

1. **work/01_eda_starter.ipynb** — Load starter CSV, check missingness, apply gotchas (avg_position=0 → NaN), save cache
2. **work/02_signal_tests.ipynb** — Load cached file, run grouped statistics (decline by has_keywords, position bucket), compute effect sizes
3. **work/03_ranking_engine.ipynb** — Build rule-based score, compute precision@K, save top-500 recommendations CSV
4. **work/capstone_paper_notebook.ipynb** (this file) — Integrate results, present findings as research paper

### Cached Outputs (in `work/outputs/`)

- `eda_starter.parquet` — Starter CSV cache (smaller, faster to load)
- `labelled_agg.parquet` — Labelled file (30k rows with decline_label)
- `labelled_with_features.parquet` — Full feature set with score_norm
- `top_timeaware_recs.csv` — Top-500 ranked recommendations (CSV, easy to share with editors)

### How to Re-run

**In Colab or Jupyter:**

1. Clone repo: `git clone https://github.com/Noorulhuda-12/flyrank`
2. Open `work/01_eda_starter.ipynb`, run all cells (loads starter CSV, saves cache)
3. Open `work/02_signal_tests.ipynb`, run all cells (if cached file present, uses it)
4. Open `work/03_ranking_engine.ipynb`, run all cells (produces recommendations)
5. Open `work/capstone_paper_notebook.ipynb`, run to regenerate this paper

**For warehouse aggregation (advanced):**
- If you have HF dataset access, the Colab cell provided in the assignment will produce `work/outputs/labelled_agg.parquet` with true time-aware labels (ctr_prev30 vs ctr_next30 around a freeze date).
- Do not commit HF token; use getpass() to supply it securely in-session only.

### Key Assumptions
- avg_position = 0 is treated as missing (not shown in results)
- Metadata is joined on (client_id, content_id)
- decline_label threshold: CTR < median for content_type (no tuning, simple proxy)
- Score weights are fixed (3, 2, 1.5, 1); not cross-validated
- No missing-value imputation; features use fillna(0) or .notna() flag

### Code Availability

All source code and notebooks are in the GitHub repository:
**https://github.com/Noorulhuda-12/flyrank**

Notebooks are designed to run end-to-end without external secrets (cached files are committed).

## Acknowledgments & Data Credit

**Built on the FlyRank ML Internship dataset**

This work uses search and engagement data provided by FlyRank (https://flyrank.ai) as part of the ML Internship program. The dataset includes anonymized content metadata, search position metrics, and click-through-rate signals from real publishers using the FlyRank platform.

**Attribution:** Crediting FlyRank as the data source is standard research practice and tells the world where this real, production data originated. Thank you to the FlyRank team for providing a rich, realistic dataset for internship capstone projects.


---

**Paper generated:** 2026-08-12

**Status:** Complete capstone, ready for review

**Next steps for production:** Validate recommendations with editors, measure action compliance, track CTR lift post-fix, iterate on score weights using performance feedback.